In [1]:
import pandas as pd
import os
from glob import glob
import re

def clean_sp2(value):
    value = re.sub(r'\(.*?\)', '', value)
    value = value.replace('NONE', '')
    return value

In [31]:
#For single runs

trainID = "F20208_heart_1Ses_time2slc_MskCrop128_3pheno_V2_3D100ep_ph3l_4ChTrans128fold0_precbf16-mixed_pythaemodel-custom_ultra_vae"
gwasID = "WBRIT_time2slc_Msk_V2_3D100ep_L1_128VAE_fullDS_ph3l"

outdir = "Output_fullDS"
# outdir = "Output_fullDS150ep"
gwasdir = "GWAS_fullDS"

path2res = f"/project/ukbblatent/Out/Results/{trainID}/{outdir}"
path2gwas = path2res.replace(outdir, gwasdir)

merged_toploci = pd.read_table(glob(f"{path2gwas}/{gwasID}/results/gwas/toploci/**/all_toploci_merged.tsv", recursive=True)[0])
print("Total number of Toploci in the merged file:", len(merged_toploci))
print("Number of Toploci with SP2:", len(merged_toploci[merged_toploci['SP2'] != 'NONE']))

merged_toploci['SP2'] = merged_toploci['SP2'].apply(clean_sp2)
merged_toploci['SP2'] = merged_toploci['SP2'].str.split(',')
merged_toploci = merged_toploci.explode('SP2')    
merged_toploci = merged_toploci[merged_toploci['SP2'] != '']
print("Total number of unique SNPs :", len(set(merged_toploci.SNP).union(set(merged_toploci.SP2))))

Total number of Toploci in the merged file: 52
Number of Toploci with SP2: 35
Total number of unique SNPs : 951


In [30]:
#Just printing the path to the GWAS data - if required to run some other code
f"{path2gwas}/{gwasID}"

'/project/ukbblatent/Out/Results/F20208_heart_1Ses_time2slc_MskCrop128_3pheno_V2_3D100ep_ph3l_4ChTrans128fold0_precbf16-mixed_pythaemodel-custom_ultra_vae/GWAS_fullDS/WBRIT_time2slc_Msk_V2_3D100ep_L1_128VAE_fullDS_ph3l'

In [56]:
#For combo comparisons (e.g. CV)

run = "Qntl_WBRIT_t2s_Msk_V2CV_3D150ep_L1_128AEL1_fullDS"
savetag = "4Interps_MagPhReIm"
path = "/group/glastonbury/GWAS/CVCNN"

merged_toploci = pd.read_table(f"{path}/{run}/all_toploci_multirun_{savetag}_merged.tsv")
print("Total number of Toploci in the merged file:", len(merged_toploci))
print("Number of Toploci with SP2:", len(merged_toploci[merged_toploci['SP2'] != 'NONE']))

merged_toploci.Run = merged_toploci.Run.str.replace(f"{run}_", "").str.replace(",","+")
merged_toploci.RunPheno = merged_toploci.RunPheno.str.replace(f"{run}_", "").str.replace(",","+")
merged_toploci.to_csv(f"{path}/{run}/all_toploci_multirun_{savetag}_merged_cleaner.tsv", sep="\t", index=False)

print("--------------------------------------------------------")

combos = set(merged_toploci.Run)
unique_runs = [r for r in combos if "+" not in r]
combo_runs = [r for r in combos if "+" in r]

str_unique_toploci = "Unique Toploci → "
str_unique_toploci_withSP2 = "Unique Toploci with SP2 → "
for run in unique_runs:
    subset = merged_toploci[merged_toploci.Run == run]
    str_unique_toploci += f"{run}: {len(subset)} | "
    str_unique_toploci_withSP2 += f"{run}: {len(subset[subset['SP2'] != 'NONE'])} | "
print(f"\n{str_unique_toploci}\n{str_unique_toploci_withSP2}\n")

str_overlapping_toploci = "Overlapping Toploci → "
str_overlapping_toploci_withSP2 = "Overlapping Toploci with SP2 → "
for run in combo_runs:
    subset = merged_toploci[merged_toploci.Run == run]
    str_overlapping_toploci += f"{run}: {len(subset)} | "
    str_overlapping_toploci_withSP2 += f"{run}: {len(subset[subset['SP2'] != 'NONE'])} | "
print(f"\n{str_overlapping_toploci}\n{str_overlapping_toploci_withSP2}\n")

print("--------------------------------------------------------")

merged_toploci['SP2'] = merged_toploci['SP2'].apply(clean_sp2)
merged_toploci['SP2'] = merged_toploci['SP2'].str.split(',')
merged_toploci = merged_toploci.explode('SP2')    
merged_toploci = merged_toploci[merged_toploci['SP2'] != '']
print("Total number of unique SNPs in the merged file :", len(set(merged_toploci.SNP).union(set(merged_toploci.SP2))))

print("--------------------------------------------------------")

str_unique_SNPs = "Unique SNPs → "
for run in unique_runs:
    subset = merged_toploci[merged_toploci.Run == run].copy()
    subset['SP2'] = subset['SP2'].apply(clean_sp2)
    subset['SP2'] = subset['SP2'].str.split(',')
    subset = subset.explode('SP2')    
    subset = subset[subset['SP2'] != '']
    str_unique_SNPs += f"{run}: {len(set(subset.SNP).union(set(subset.SP2)))} | "
print(f"\n{str_unique_SNPs}\n")

str_overlapping_SNPs = "Overlapping SNPs → "
for run in combo_runs:
    subset = merged_toploci[merged_toploci.Run == run].copy()
    subset['SP2'] = subset['SP2'].apply(clean_sp2)
    subset['SP2'] = subset['SP2'].str.split(',')
    subset = subset.explode('SP2')    
    subset = subset[subset['SP2'] != '']
    str_overlapping_SNPs += f"{run}: {len(set(subset.SNP).union(set(subset.SP2)))} | "
print(f"\n{str_overlapping_SNPs}\n")

Total number of Toploci in the merged file: 191
Number of Toploci with SP2: 125
--------------------------------------------------------

Unique Toploci → phase: 51 | real: 32 | imag: 38 | mag: 39 | 
Unique Toploci with SP2 → phase: 35 | real: 17 | imag: 25 | mag: 23 | 


Overlapping Toploci → mag+real: 1 | real+phase: 3 | imag+mag: 8 | imag+phase: 5 | real+imag+mag+phase: 1 | phase+imag+mag: 1 | real+mag: 11 | phase+mag: 1 | 
Overlapping Toploci with SP2 → mag+real: 1 | real+phase: 2 | imag+mag: 5 | imag+phase: 5 | real+imag+mag+phase: 0 | phase+imag+mag: 1 | real+mag: 10 | phase+mag: 1 | 

--------------------------------------------------------
Total number of unique SNPs in the merged file : 4134
--------------------------------------------------------

Unique SNPs → phase: 338 | real: 439 | imag: 753 | mag: 385 | 


Overlapping SNPs → mag+real: 34 | real+phase: 63 | imag+mag: 1795 | imag+phase: 114 | real+imag+mag+phase: 0 | phase+imag+mag: 5 | real+mag: 202 | phase+mag: 7 | 

